# 🤖 RAG-система на Qwen2.5-3B-Instruct + FAISS + LangChain

**Архитектура:**
```
Arxiv (100 юридических статей)
        │
        ▼
  Recursive Chunking
        │
        ▼
  HuggingFace Embeddings
        │
        ▼
  FAISS (in-memory)
        │
        ▼
  RetrievalQA Agent (Qwen2.5-3B-Instruct)
        │
        ▼
     Ответ
```

## 📦 Ячейка 1 — Установка зависимостей

In [21]:
# Устанавливаем все необходимые библиотеки
# langchain-text-splitters — отдельный пакет начиная с LangChain v0.2
!pip install langchain langchain-community langchain-huggingface langchain-text-splitters langchain-core
!pip install faiss-cpu
!pip install transformers accelerate
!pip install arxiv
!pip install sentence-transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# Если нет CUDA — используйте CPU версию:
# !pip install torch torchvision torchaudio


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu118



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 📚 Ячейка 2 — Импорты

In [22]:
import os
import arxiv
import torch
import warnings
warnings.filterwarnings('ignore')

# ── LangChain v0.2+ — все модули в отдельных пакетах ─────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

# langchain_core — базовые примитивы (Document, Prompt, LCEL)
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# HuggingFace Transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
)

print("✅ Все импорты успешны")
print(f"🔥 CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅ Все импорты успешны
🔥 CUDA доступна: True
   GPU: NVIDIA GeForce RTX 3060
   VRAM: 12.9 GB


## 🌐 Ячейка 3 — Загрузка 100 юридических статей с Arxiv

In [23]:
# ── Установка datasets (если ещё нет) ───────────────────────────────────────
!pip install datasets -q

from datasets import load_dataset
from langchain_core.documents import Document

# ── Выбор датасета ────────────────────────────────────────────────────────────
# Меняй DATASET_NAME и CONFIG чтобы переключаться между корпусами:
#
#  ВАРИАНТ 1 — LEDGAR (контрактные положения из SEC, ~60k записей)
#    name="coastalcph/lex_glue", config="ledgar"
#    Поля: text (положение договора), label (тип: indemnification, termination...)
#
#  ВАРИАНТ 2 — ECtHR (решения Европейского суда по правам человека)
#    name="coastalcph/lex_glue", config="ecthr_a"
#    Поля: text (список фактов дела), labels (нарушенные статьи ЕКПЧ)
#
#  ВАРИАНТ 3 — EUR-Lex (директивы и регламенты ЕС, многоуровневые метки)
#    name="coastalcph/lex_glue", config="eurlex"
#    Поля: text (текст документа), labels (коды EUROVOC)
#
#  ВАРИАНТ 4 — SCOTUS (решения Верховного суда США)
#    name="coastalcph/lex_glue", config="scotus"
#    Поля: text (мнение суда), label (область права)
#
#  ВАРИАНТ 5 — MultiLegalPile (огромный корпус, 24 юрисдикции)
#    name="joelito/MultiLegalPile_en", config=None, streaming=True

DATASET_NAME   = "coastalcph/lex_glue"
DATASET_CONFIG = "ledgar"      # ← меняй здесь
NUM_DOCS       = 200           # ← сколько документов загрузить

# ── Маппинг числовых меток LEDGAR → читаемые названия ────────────────────────
LEDGAR_LABELS = {
    0: "Adjustments", 1: "Agreements", 2: "Amendments", 3: "Anti-Corruption",
    4: "Applicable Laws", 5: "Approvals", 6: "Arbitration", 7: "Assignments",
    8: "Assigns", 9: "Authority", 10: "Authorizations", 11: "Base Salary",
    12: "Benefits", 13: "Books", 14: "Brokers", 15: "Change Of Control",
    16: "Closings", 17: "Compliance With Laws", 18: "Confidentiality",
    19: "Consent To Jurisdiction", 20: "Consents", 21: "Construction",
    22: "Cooperation", 23: "Costs", 24: "Counterparts", 25: "Death",
    26: "Defined Terms", 27: "Definitions", 28: "Disability",
    29: "Disclosures", 30: "Duties", 31: "Effective Dates",
    32: "Effectiveness", 33: "Employment", 34: "Enforceability",
    35: "Entire Agreements", 36: "Erisa", 37: "Existence",
    38: "Expenses", 39: "Fees", 40: "Financial Statements",
    41: "Further Assurances", 42: "General", 43: "Governing Laws",
    44: "Headings", 45: "Indemnification", 46: "Insurances",
    47: "Integration", 48: "Intellectual Property", 49: "Interest",
    50: "Interpretations", 51: "Jurisdictions", 52: "Liens",
    53: "Litigations", 54: "Miscellaneous", 55: "Modifications",
    56: "Non-Compete", 57: "Non-Disparagement", 58: "Notices",
    59: "Organizations", 60: "Payments", 61: "Positions",
    62: "Powers", 63: "Publicity", 64: "Qualifications",
    65: "Records", 66: "Releases", 67: "Remedies",
    68: "Representations", 69: "Sales", 70: "Sanctions",
    71: "Severability", 72: "Solvency", 73: "Specific Performance",
    74: "Successors", 75: "Survival", 76: "Tax Withholdings",
    77: "Taxes", 78: "Termination", 79: "Terms",
    80: "Titles", 81: "Transactions With Affiliates", 82: "Use Of Proceeds",
    83: "Venues", 84: "Waivers", 85: "Warranties", 86: "Work Hours",
    87: "No-Defaults", 88: "No Material Adverse Change", 89: "Participations",
    90: "Penalties", 91: "Representations And Warranties",
    92: "Third Party Beneficiaries", 93: "Withholdings",
}

# ── Загрузка ──────────────────────────────────────────────────────────────────
print(f"🔄 Загружаем датасет: {DATASET_NAME} / {DATASET_CONFIG}")
print(f"   Количество документов: {NUM_DOCS}\n")

ds = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=f"train[:{NUM_DOCS}]",
    trust_remote_code=True
)

# ── Конвертируем в LangChain Documents ───────────────────────────────────────
unique_documents = []
for row in ds:
    # Для LEDGAR: text — само положение, label — числовой индекс типа
    label_id  = row.get("label", row.get("labels", 0))
    # Если метка — список (ecthr, eurlex), берём первую
    if isinstance(label_id, list):
        label_id = label_id[0] if label_id else 0
    label_name = LEDGAR_LABELS.get(int(label_id), f"Category_{label_id}")

    doc = Document(
        page_content=row["text"],
        metadata={
            "title":   f"[{label_name}] {row['text'][:60].strip()}...",
            "label":   label_name,
            "label_id": int(label_id),
            "source":  f"{DATASET_NAME}/{DATASET_CONFIG}",
            "authors": ["LEDGAR/SEC corpus"],
            "published": "2023",
            "url": "https://huggingface.co/datasets/coastalcph/lex_glue",
        }
    )
    unique_documents.append(doc)

# ── Статистика ────────────────────────────────────────────────────────────────
from collections import Counter
label_counts = Counter(d.metadata["label"] for d in unique_documents)

print(f"✅ Загружено: {len(unique_documents)} документов")
print(f"\n📊 Топ-10 типов контрактных положений:")
for label, count in label_counts.most_common(10):
    bar = '█' * count
    print(f"   {label:<35} {count:>3}  {bar}")

print(f"\n📌 Пример документа:")
print("-" * 60)
print(f"Тип:    {unique_documents[0].metadata['label']}")
print(f"Текст:  {unique_documents[0].page_content[:300]}")
print("-" * 60)



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'coastalcph/lex_glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


🔄 Загружаем датасет: coastalcph/lex_glue / ledgar
   Количество документов: 200

✅ Загружено: 200 документов

📊 Топ-10 типов контрактных положений:
   Expenses                             13  █████████████
   Defined Terms                        11  ███████████
   Integration                          10  ██████████
   Records                               8  ████████
   Terms                                 8  ████████
   Assignments                           6  ██████
   Warranties                            6  ██████
   Further Assurances                    5  █████
   No Material Adverse Change            5  █████
   Category_97                           4  ████

📌 Пример документа:
------------------------------------------------------------
Тип:    Category_97
Текст:  Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment,

## ✂️ Ячейка 4 — Recursive Chunking

In [24]:
# ── Recursive Character Text Splitter ────────────────────────────────────────
# Рекурсивно делит текст по: параграфам -> предложениям -> словам -> символам
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,          # Размер чанка в символах
    chunk_overlap=64,        # Перекрытие для сохранения контекста
    length_function=len,
    separators=[
        "\n\n",  # Параграфы (приоритет 1)
        "\n",    # Строки (приоритет 2)
        ". ",    # Предложения (приоритет 3)
        " ",     # Слова (приоритет 4)
        "",      # Символы (приоритет 5 — крайний случай)
    ],
    is_separator_regex=False
)

# Разбиваем документы на чанки
chunks = text_splitter.split_documents(unique_documents)

print(f"📊 Статистика чанкинга:")
print(f"   Документов до разбивки: {len(unique_documents)}")
print(f"   Чанков после разбивки:  {len(chunks)}")
print(f"   Среднее чанков/документ: {len(chunks)/len(unique_documents):.1f}")

# Статистика по размерам чанков
chunk_sizes = [len(c.page_content) for c in chunks]
print(f"\n📏 Размеры чанков:")
print(f"   Минимальный: {min(chunk_sizes)} символов")
print(f"   Максимальный: {max(chunk_sizes)} символов")
print(f"   Средний: {sum(chunk_sizes)/len(chunk_sizes):.0f} символов")

print(f"\n📌 Пример чанка:")
print("-" * 60)
print(chunks[5].page_content)
print("-" * 60)
print(f"Метаданные: {chunks[5].metadata}")

📊 Статистика чанкинга:
   Документов до разбивки: 200
   Чанков после разбивки:  423
   Среднее чанков/документ: 2.1

📏 Размеры чанков:
   Минимальный: 60 символов
   Максимальный: 512 символов
   Средний: 365 символов

📌 Пример чанка:
------------------------------------------------------------
Commencing March 7, 2016 and during the Employment Period, the Company shall pay to the Executive a base salary at the rate of no less than $750,000 per calendar year (the “Base Salary”), less applicable deductions, and prorated for any partial month or year, as applicable. The Base Salary shall be reviewed for increase by the Compensation Committees of AFG and AAC (the “Compensation Committees”) no less frequently than annually and may be increased in the discretion of the Compensation Committees
------------------------------------------------------------
Метаданные: {'title': '[Base Salary] Commencing March 7, 2016 and during the Employment Period, t...', 'label': 'Base Salary', 'label_id': 

## 🧬 Ячейка 5 — Embeddings + FAISS Vector Store

In [25]:
# ── Embedding Model ───────────────────────────────────────────────────────────
# Используем лёгкую модель эмбеддингов (работает на CPU и GPU)
EMBEDDING_MODEL_NAME = "D:/bogdanov/PyProjects/Agent_system1/Models/embeddings"

print(f"🔄 Загружаем embedding модель: {EMBEDDING_MODEL_NAME}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},  # Косинусное сходство
)

print(f"✅ Embedding модель загружена")
print(f"   Устройство: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

# ── FAISS Vector Store ────────────────────────────────────────────────────────
print(f"\n🗄️  Создаём FAISS индекс (in-memory)...")
print(f"   Индексируем {len(chunks)} чанков...")

# Батчевая индексация для скорости
BATCH_SIZE = 64
vectorstore = None

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    
    if vectorstore is None:
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        vectorstore.add_documents(batch)
    
    print(f"   ✓ Обработано {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)} чанков")

print(f"\n✅ FAISS индекс создан!")
print(f"   Векторов в базе: {vectorstore.index.ntotal}")
print(f"   Размерность векторов: {vectorstore.index.d}")

🔄 Загружаем embedding модель: D:/bogdanov/PyProjects/Agent_system1/Models/embeddings
✅ Embedding модель загружена
   Устройство: CUDA

🗄️  Создаём FAISS индекс (in-memory)...
   Индексируем 423 чанков...
   ✓ Обработано 64/423 чанков
   ✓ Обработано 128/423 чанков
   ✓ Обработано 192/423 чанков
   ✓ Обработано 256/423 чанков
   ✓ Обработано 320/423 чанков
   ✓ Обработано 384/423 чанков
   ✓ Обработано 423/423 чанков

✅ FAISS индекс создан!
   Векторов в базе: 423
   Размерность векторов: 1024


## 🔍 Ячейка 6 — Тест Retriever

In [26]:
# Настраиваем retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",          # Maximum Marginal Relevance — разнообразие результатов
    search_kwargs={
        "k": 4,                 # Топ-4 документа
        "fetch_k": 20,          # Из 20 кандидатов
        "lambda_mult": 0.7      # 0=максимальное разнообразие, 1=максимальная релевантность
    }
)

# ── Тестовый запрос ───────────────────────────────────────────────────────────
test_query = "How is artificial intelligence regulated in legal systems?"

print(f"🔍 Тестовый запрос: '{test_query}'\n")
print("=" * 70)

retrieved_docs = retriever.invoke(test_query)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n📄 Документ #{i}")
    print(f"   Название: {doc.metadata.get('title', 'N/A')[:70]}")
    print(f"   Авторы:   {', '.join(doc.metadata.get('authors', [])[:2])}")
    print(f"   Дата:     {doc.metadata.get('published', 'N/A')}")
    print(f"   Фрагмент: {doc.page_content[:200]}...")
    print("-" * 70)

🔍 Тестовый запрос: 'How is artificial intelligence regulated in legal systems?'


📄 Документ #1
   Название: [Records] Each Party shall immediately provide the other Party with wr
   Авторы:   LEDGAR/SEC corpus
   Дата:     2023
   Фрагмент: a Third Party, (ii) “patent certification” filed in the United States under 21 U.S.C...
----------------------------------------------------------------------

📄 Документ #2
   Название: [Authorizations] The Seller represents and Certificates that all actio
   Авторы:   LEDGAR/SEC corpus
   Дата:     2023
   Фрагмент: and other laws of general application affecting enforcement of creditor’s rights generally and to general equitable principles...
----------------------------------------------------------------------

📄 Документ #3
   Название: [Arbitration] Any and all Arbitrable Disputes (except to the extent in
   Авторы:   LEDGAR/SEC corpus
   Дата:     2023
   Фрагмент: . If there is any inconsistency between this Article 22 and the Commercial A

## 🤖 Ячейка 7 — Загрузка Qwen2.5-3B-Instruct

In [27]:
# ── Путь к локальной модели ───────────────────────────────────────────────────
MODEL_PATH = r"D:\bogdanov\PyProjects\Agent_system1\Models\Qwen2.5-3B-Instruct"

# Проверяем что путь существует
assert os.path.exists(MODEL_PATH), f"❌ Модель не найдена по пути: {MODEL_PATH}"
print(f"✅ Модель найдена: {MODEL_PATH}")

# ── Определяем устройство ─────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔧 Устройство: {DEVICE}")

# ── Загружаем токенизатор ─────────────────────────────────────────────────────
print("\n🔄 Загружаем токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)
# Устанавливаем pad_token если его нет
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Токенизатор загружен")

# ── Загружаем модель ──────────────────────────────────────────────────────────
print("\n🔄 Загружаем Qwen2.5-3B-Instruct...")

if DEVICE == "cuda":
    # GPU: загружаем в half-precision для экономии VRAM
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16,
        device_map="auto",          # Автоматически распределяет по GPU/CPU
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
else:
    # CPU: загружаем в float32
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float32,
        device_map="cpu",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )

model.eval()
print("✅ Модель загружена!")

# Память
if DEVICE == "cuda":
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    print(f"   VRAM использовано:   {allocated:.2f} GB")
    print(f"   VRAM зарезервировано: {reserved:.2f} GB")

✅ Модель найдена: D:\bogdanov\PyProjects\Agent_system1\Models\Qwen2.5-3B-Instruct
🔧 Устройство: cuda

🔄 Загружаем токенизатор...
✅ Токенизатор загружен

🔄 Загружаем Qwen2.5-3B-Instruct...


Loading checkpoint shards: 100%|██████████| 2/2 [00:24<00:00, 12.44s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


✅ Модель загружена!
   VRAM использовано:   11.51 GB
   VRAM зарезервировано: 11.66 GB


## 🔗 Ячейка 8 — Создание HuggingFace Pipeline для LangChain

In [28]:
# ── HuggingFace Pipeline ──────────────────────────────────────────────────────
hf_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,          # Максимальная длина ответа
    temperature=0.1,             # Низкая для фактических ответов
    do_sample=True,
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.15,     # Уменьшаем повторения
    return_full_text=False,      # Возвращаем только новый текст
    pad_token_id=tokenizer.eos_token_id,
)

# ── Оборачиваем в LangChain LLM ───────────────────────────────────────────────
llm = HuggingFacePipeline(
    pipeline=hf_pipeline,
    model_kwargs={"temperature": 0.1}
)

print("✅ LangChain LLM готов!")

# Быстрый тест LLM напрямую
print("\n🧪 Быстрый тест LLM (без RAG):")
test_response = llm.invoke("What is the role of AI in legal research? Answer briefly.")
print(f"Ответ: {test_response[:300]}..." if len(test_response) > 300 else f"Ответ: {test_response}")

Device set to use cuda:0


✅ LangChain LLM готов!

🧪 Быстрый тест LLM (без RAG):
Ответ:  Artificial Intelligence (AI) plays a significant role in enhancing and streamlining legal research by automating tasks, improving search efficiency, providing predictive analytics, and supporting document review processes. It helps lawyers find relevant cases faster and more accurately, reducing ti...


## 🎯 Ячейка 9 — Создание RAG Prompt Template

In [29]:
# ── Промпт в стиле Qwen2.5 Instruct ──────────────────────────────────────────
RAG_PROMPT_TEMPLATE = """<|im_start|>system
You are a helpful legal research assistant. Answer questions based strictly on the provided context from academic papers. 
If the context does not contain enough information, say so clearly.
Be concise, accurate and cite relevant details from the papers when possible.<|im_end|>
<|im_start|>user
Context from academic papers:
-----
{context}
-----

Question: {question}

Please provide a detailed answer based on the context above.<|im_end|>
<|im_start|>assistant
"""

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=RAG_PROMPT_TEMPLATE
)

print("✅ RAG Prompt Template создан")
print("\n📋 Шаблон промпта:")
print("-" * 60)
print(RAG_PROMPT_TEMPLATE.replace("{context}", "[RETRIEVED CHUNKS]").replace("{question}", "[USER QUESTION]"))
print("-" * 60)

✅ RAG Prompt Template создан

📋 Шаблон промпта:
------------------------------------------------------------
<|im_start|>system
You are a helpful legal research assistant. Answer questions based strictly on the provided context from academic papers. 
If the context does not contain enough information, say so clearly.
Be concise, accurate and cite relevant details from the papers when possible.<|im_end|>
<|im_start|>user
Context from academic papers:
-----
[RETRIEVED CHUNKS]
-----

Question: [USER QUESTION]

Please provide a detailed answer based on the context above.<|im_end|>
<|im_start|>assistant

------------------------------------------------------------


## ⛓️ Ячейка 10 — Сборка RAG Chain

In [30]:
# ── LCEL RAG Chain (замена устаревшего RetrievalQA) ──────────────────────────
# В LangChain v0.2+ RetrievalQA убран — используем LCEL (pipe-синтаксис)

def format_docs(docs: list) -> str:
    """Склеиваем retrieved чанки в одну строку контекста."""
    return "\n\n".join(
        f"[Источник: {d.metadata.get('title', 'N/A')[:60]}]\n{d.page_content}"
        for d in docs
    )

# Цепочка:
#   вопрос → retriever → format_docs → prompt → llm → парсер строки
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✅ LCEL RAG Chain собран!")
print("""
🏗️  Компоненты системы:
   📚 Vector Store:  FAISS (in-memory)
   🧬 Embeddings:    sentence-transformers/all-MiniLM-L6-v2
   🔍 Retriever:     MMR, top-4 из 20 кандидатов
   🤖 LLM:           Qwen2.5-3B-Instruct (локальная)
   ⛓️  Chain:         LCEL (Runnable pipe)
""")

✅ LCEL RAG Chain собран!

🏗️  Компоненты системы:
   📚 Vector Store:  FAISS (in-memory)
   🧬 Embeddings:    sentence-transformers/all-MiniLM-L6-v2
   🔍 Retriever:     MMR, top-4 из 20 кандидатов
   🤖 LLM:           Qwen2.5-3B-Instruct (локальная)
   ⛓️  Chain:         LCEL (Runnable pipe)



## 🚀 Ячейка 11 — Вспомогательная функция для запросов

In [31]:
def ask_rag(question: str, show_sources: bool = True) -> str:
    """
    Задаёт вопрос RAG-системе и возвращает ответ с источниками.
    LCEL-цепочка возвращает только строку ответа;
    источники получаем отдельным вызовом retriever.invoke().
    """
    print("=" * 70)
    print(f"❓ ВОПРОС: {question}")
    print("=" * 70)
    print("🔄 Обрабатываем запрос...\n")

    # Получаем ответ
    answer = rag_chain.invoke(question)

    # Чистим от служебных токенов Qwen
    for stop in ["<|im_end|>", "<|im_start|>", "Question:", "Context:"]:
        if stop in answer:
            answer = answer.split(stop)[0]
    answer = answer.strip()

    print("💬 ОТВЕТ:")
    print("-" * 70)
    print(answer)

    if show_sources:
        # Отдельно получаем источники
        source_docs = retriever.invoke(question)
        if source_docs:
            print("\n" + "=" * 70)
            print("📚 ИСПОЛЬЗОВАННЫЕ ИСТОЧНИКИ:")
            print("=" * 70)
            seen = set()
            for i, doc in enumerate(source_docs, 1):
                title = doc.metadata.get('title', 'N/A')
                if title not in seen:
                    seen.add(title)
                    print(f"  [{i}] {title[:75]}")
                    print(f"      Авторы: {', '.join(doc.metadata.get('authors', [])[:2])}")
                    print(f"      Дата:   {doc.metadata.get('published', 'N/A')}")
                    print(f"      URL:    {doc.metadata.get('url', 'N/A')}")
                    print()

    return answer


print("✅ Функция ask_rag() готова к использованию!")

✅ Функция ask_rag() готова к использованию!


## 🧪 Ячейка 12 — Тестирование RAG-системы

In [32]:
# ── Тест 1: Регулирование ИИ в праве ─────────────────────────────────────────
answer1 = ask_rag(
    "How is artificial intelligence being regulated in legal systems?"
)

❓ ВОПРОС: How is artificial intelligence being regulated in legal systems?
🔄 Обрабатываем запрос...

💬 ОТВЕТ:
----------------------------------------------------------------------
The given context pertains specifically to corporate governance, intellectual property disputes, employment agreements, patent certifications, and seller representations rather than regulations related to artificial intelligence (AI). Therefore, there is insufficient information directly addressing how AI is being regulated in legal systems from the provided sources.

However, it's important to note that while these contexts don't explicitly discuss AI regulation, they highlight areas where technology might influence future regulatory frameworks:

- **Intellectual Property**: As AI generates content and inventions, issues around copyright, patents, and trademarks will likely become more complex. For instance, determining who owns the IP generated through AI could require new legislation.
  
- **Employment La

In [33]:
# ── Тест 2: NLP в судопроизводстве ───────────────────────────────────────────
answer2 = ask_rag(
    "What are the main applications of NLP and machine learning in court proceedings?"
)

❓ ВОПРОС: What are the main applications of NLP and machine learning in court proceedings?
🔄 Обрабатываем запрос...

💬 ОТВЕТ:
----------------------------------------------------------------------
The given context pertains specifically to contractual agreements regarding litigation matters rather than directly addressing Natural Language Processing (NLP) and Machine Learning (ML). However, we can infer some indirect implications:

1. **Contractual Agreements**: There's mention of "Transaction Documents" which likely include contracts involving various parties' responsibilities and obligations. These documents could potentially benefit from text analysis tools like NLP.

2. **Jurisdictional Disputes**: The agreement specifies that any legal proceedings related to these documents must be filed in specific U.S. courts. This might involve automated systems analyzing contract language to predict potential jurisdictional issues before they arise.

3. **Dispute Resolution Mechanisms**: While

In [34]:
# ── Тест 3: GDPR и конфиденциальность ────────────────────────────────────────
answer3 = ask_rag(
    "What challenges does GDPR pose for machine learning and data privacy?"
)

❓ ВОПРОС: What challenges does GDPR pose for machine learning and data privacy?
🔄 Обрабатываем запрос...

💬 ОТВЕТ:
----------------------------------------------------------------------
The provided context is unrelated to GDPR and does not address any specific challenges it poses for machine learning or data privacy. Therefore, there isn't sufficient information within the given text to directly answer the question about GDPR's impact on these areas.

However, I can highlight some general points related to GDPR:

1. **Data Privacy**: GDPR emphasizes stringent rules regarding personal data protection and requires organizations to obtain explicit consent before processing sensitive data. This aligns closely with the confidentiality agreements mentioned in one part of the provided context where parties must keep certain data confidential.

2. **Transparency**: GDPR mandates transparency concerning how user data is used and shared. Organizations need clear policies outlining their practic

In [35]:
# ── Тест 4: Справедливость алгоритмов ────────────────────────────────────────
answer4 = ask_rag(
    "What are the fairness concerns with algorithmic decision-making in criminal justice?"
)

❓ ВОПРОС: What are the fairness concerns with algorithmic decision-making in criminal justice?
🔄 Обрабатываем запрос...

💬 ОТВЕТ:
----------------------------------------------------------------------
The provided context is unrelated to the question about fairness concerns regarding algorithmic decision-making in criminal justice. Therefore, there is insufficient information within the given context to address the specific question asked.

📚 ИСПОЛЬЗОВАННЫЕ ИСТОЧНИКИ:
  [1] [Authorizations] The Seller represents and Certificates that all action on 
      Авторы: LEDGAR/SEC corpus
      Дата:   2023
      URL:    https://huggingface.co/datasets/coastalcph/lex_glue

  [2] [Effective Dates] The Borrower has disclosed to the Lenders all material ag
      Авторы: LEDGAR/SEC corpus
      Дата:   2023
      URL:    https://huggingface.co/datasets/coastalcph/lex_glue

  [3] [Arbitration] Any dispute, controversy, claim or action of any kind arisin.
      Авторы: LEDGAR/SEC corpus
      Дата:  

## 💾 Ячейка 13 — (Опционально) Сохранение FAISS индекса на диск

In [36]:
# Можно сохранить индекс чтобы не пересоздавать при каждом запуске
FAISS_INDEX_PATH = r"D:\bogdanov\PyProjects\Agents_project\faiss_legal_index"

# ── Сохранение ─────────────────────────────────────────────────────────────────
vectorstore.save_local(FAISS_INDEX_PATH)
print(f"✅ FAISS индекс сохранён: {FAISS_INDEX_PATH}")

# ── Загрузка (при следующем запуске) ──────────────────────────────────────────
# vectorstore_loaded = FAISS.load_local(
#     FAISS_INDEX_PATH,
#     embeddings,
#     allow_dangerous_deserialization=True
# )
# print(f"✅ FAISS индекс загружен: {vectorstore_loaded.index.ntotal} векторов")

✅ FAISS индекс сохранён: D:\bogdanov\PyProjects\Agents_project\faiss_legal_index


## 🎮 Ячейка 14 — Интерактивный режим

In [37]:
# Интерактивный цикл вопрос-ответ
# Введите 'exit' для выхода

print("🎮 Интерактивный режим RAG-системы")
print("   Введите вопрос на английском языке")
print("   Введите 'exit' для выхода\n")

while True:
    user_input = input("\n🙋 Ваш вопрос: ").strip()
    
    if user_input.lower() in ["exit", "quit", "выход", "q"]:
        print("👋 Выход из интерактивного режима")
        break
    
    if not user_input:
        print("⚠️  Пустой вопрос, попробуйте ещё раз")
        continue
    
    ask_rag(user_input, show_sources=True)

🎮 Интерактивный режим RAG-системы
   Введите вопрос на английском языке
   Введите 'exit' для выхода

👋 Выход из интерактивного режима


## 📊 Ячейка 15 — Статистика системы

In [38]:
print("📊 ИТОГОВАЯ СТАТИСТИКА RAG-СИСТЕМЫ")
print("=" * 50)
print(f"📚 Документов в базе:      {len(unique_documents)}")
print(f"✂️  Чанков в индексе:       {len(chunks)}")
print(f"🗄️  Векторов в FAISS:       {vectorstore.index.ntotal}")
print(f"🧬 Размерность эмбеддингов: {vectorstore.index.d}")
print(f"🤖 LLM:                    Qwen2.5-3B-Instruct")
print(f"💾 Устройство:             {DEVICE.upper()}")
if torch.cuda.is_available():
    print(f"🔥 GPU:                    {torch.cuda.get_device_name(0)}")
    print(f"   VRAM:                   {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("=" * 50)
print("✅ Система готова к работе!")

📊 ИТОГОВАЯ СТАТИСТИКА RAG-СИСТЕМЫ
📚 Документов в базе:      200
✂️  Чанков в индексе:       423
🗄️  Векторов в FAISS:       423
🧬 Размерность эмбеддингов: 1024
🤖 LLM:                    Qwen2.5-3B-Instruct
💾 Устройство:             CUDA
🔥 GPU:                    NVIDIA GeForce RTX 3060
   VRAM:                   5.34 GB
✅ Система готова к работе!
